
# Optimized BioCatch Vishing Data Augmentation Pipeline

Versión optimizada para ejecución en Kaggle P100/T4 con enfoque en:

- reducción de consumo RAM
- generación chunked
- exportación incremental
- entrenamiento progresivo
- control de memoria
- parquet streaming
- validaciones eficientes

Objetivo:
- generar ~1M observaciones
- mantener fraude ~1%
- evitar OOM en Kaggle


In [ ]:
# DIAGNÓSTICO GPU 

import torch, subprocess, sys

print(f"torch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap  = torch.cuda.get_device_capability(0)
    print(f"GPU detectada : {name}")
    print(f"Compute cap.  : sm_{cap[0]}{cap[1]}")

    # Smoke-test real
    try:
        t = torch.zeros(4, 4).cuda()
        _ = t @ t
        del t
        torch.cuda.empty_cache()
        print("\n✅ GPU OPERATIVA — puedes correr el notebook completo")
    except Exception as e:
        sm = f"sm_{cap[0]}{cap[1]}"
        print(f"\n❌ GPU FALLA ({sm}): {e}")
        if cap[0] < 7:
            print(f"\n{'='*56}")
            print(f"  {name} ({sm}) es incompatible con Python 3.12 + torch 2.x")
            print(f"  PyTorch dejó de soportar sm_60 en torch 2.0 (Feb 2023).")
            print(f"  Python 3.12 fue lanzado en Oct 2023 — nunca se solaparon.")
            print(f"")
            print(f"  ► Cambiar acelerador: Settings > Accelerator > GPU T4 x1")
            print(f"  T4 = sm_75, 16 GB VRAM, compatible con torch 2.x + Python 3.12")
            print(f"{'='*56}")
        else:
            print("Error inesperado. Verifica los logs de CUDA.")
else:
    print("\n⚠️  Sin GPU. Activa el acelerador en Settings > Accelerator > GPU T4 x1")


torch: 2.10.0+cu128
CUDA disponible: True
GPU detectada : Tesla T4
Compute cap.  : sm_75

✅ GPU OPERATIVA — puedes correr el notebook completo


In [2]:

# ====================================
# INSTALLS (KAGGLE)
# ====================================
# sdv 1.36 requiere ctgan >= 0.10; ctgan 0.10+ funciona con torch 2.1
!pip install -q sdv pyarrow fastparquet psutil


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.8/204.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 52.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.5 MB/s eta 0:00:0000:01


In [4]:
import warnings
warnings.filterwarnings("ignore")

import gc
import os
import random
import sys
import time
import inspect
import subprocess

import numpy as np
import pandas as pd
import psutil
import torch
import sdv

import pyarrow as pa
import pyarrow.parquet as pq

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer

In [ ]:

# ====================================
# CONFIG
# ====================================

DATASET_PATH = "/kaggle/input/datasets/jm0ntyy/dataset-sintetico-biocatch-vishing-overlap/dataset_sintetico_biocatch_vishing_overlap.csv"

OUTPUT_PARQUET = "/kaggle/working/biocatch_augmented_1M.parquet"

TARGET_ROWS = 1_000_000
TARGET_VISHING = 10_000
TARGET_LEGIT = TARGET_ROWS - TARGET_VISHING

CHUNK_SIZE = 100_000

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ====================================
# GPU / PYTORCH ENV SETUP
# ====================================

# Limitar fragmentación de memoria en la GPU
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"


# ====================================
# MEMORY HELPERS
# ====================================

def ram_free_gb():
    return psutil.virtual_memory().available / 1024**3

def vram_free_gb():
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info(0)
        return free / 1024**3
    return 0.0

def print_resources(label=""):
    prefix = f"[{label}] " if label else ""
    ram = psutil.virtual_memory()
    msg = (
        f"{prefix}RAM: {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f} GB "
        f"({ram.percent:.0f}% usado)"
    )
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info(0)
        msg += f" | VRAM libre: {free/1024**3:.2f}/{total/1024**3:.2f} GB"
    print(msg)

def aggressive_gc():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

RAM_LOW_THRESHOLD_GB = 2.0   # pausar si RAM libre cae por debajo de esto
VRAM_LOW_THRESHOLD_GB = 1.5  # advertencia si VRAM libre cae por debajo de esto

print("Config cargada.")
print_resources("INICIO")


Config cargada.
[INICIO] RAM: 1.4/31.4 GB (6% usado) | VRAM libre: 14.43/14.56 GB


In [6]:

# ====================================
# LOAD DATASET
# ====================================

df = pd.read_csv(DATASET_PATH)

print(df.shape)

df.head()


(50000, 61)


,session_id,customer_id,session_timestamp,device_type,os_type,app_version,avg_keyhold_ms,avg_interkey_latency_ms,typing_speed_cps,keystroke_variability,...,biocatch_genuine_score,biocatch_ato_indicator,biocatch_social_eng_indicator,biocatch_bot_indicator,errors_per_minute,interactions_per_s,hesitation_composite,is_vishing,days_to_claim,claim_category
0,SES-NO-000001,CUS-53613,2024-08-24 14:54:24,mobile,iOS,v13.0,97.7,178.4,3.777,0.240,...,896,0,0,0,8.717,2.4,1.1518,0,-1,none
1,SES-NO-000002,CUS-43723,2025-02-06 21:28:41,mobile,iOS,v12.3,72.8,78.5,3.016,0.086,...,670,1,0,0,8.717,2.4,1.0316,0,-1,none
2,SES-NO-000003,CUS-93608,2025-05-15 15:22:09,web,Windows,v12.3,86.7,152.3,4.776,0.141,...,838,0,0,0,0.000,2.4,1.1518,0,-1,none
3,SES-NO-000004,CUS-25829,2024-08-08 06:00:27,mobile,Android,v13.0,82.9,176.7,4.992,0.171,...,570,0,0,0,8.717,2.4,1.1518,0,-1,none
4,SES-NO-000005,CUS-46701,2024-09-04 11:09:02,mobile,Android,v12.1,69.9,204.0,6.337,0.133,...,1000,0,0,0,8.717,2.4,1.1518,0,-1,none


In [7]:

# ====================================
# MEMORY OPTIMIZATION
# ====================================

def optimize_memory(df):

    for col in df.columns:

        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')

        elif df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')

        elif df[col].dtype == 'object':

            nunique = df[col].nunique()

            if nunique / len(df) < 0.5:
                df[col] = df[col].astype('category')

    return df

df = optimize_memory(df)

df.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 61 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   session_id                     50000 non-null  object  
 1   customer_id                    50000 non-null  category
 2   session_timestamp              50000 non-null  object  
 3   device_type                    50000 non-null  category
 4   os_type                        50000 non-null  category
 5   app_version                    50000 non-null  category
 6   avg_keyhold_ms                 50000 non-null  float32 
 7   avg_interkey_latency_ms        50000 non-null  float32 
 8   typing_speed_cps               50000 non-null  float32 
 9   keystroke_variability          50000 non-null  float32 
 10  segmented_typing_ratio         50000 non-null  float32 
 11  avg_touch_pressure             50000 non-null  float32 
 12  avg_touch_size_px              5

In [8]:

# ====================================
# REMOVE LEAKAGE
# ====================================

leakage_cols = [
    'biocatch_risk_score',
    'biocatch_genuine_score',
    'biocatch_ato_indicator',
    'biocatch_social_eng_indicator',
    'biocatch_bot_indicator',
    'claim_category',
    'days_to_claim'
]

existing_leakage = [
    c for c in leakage_cols if c in df.columns
]

df_model = df.drop(columns=existing_leakage)

print(df_model.shape)


(50000, 54)


In [9]:

# ====================================
# SPLIT CLASSES
# ====================================

legit_df = df_model[df_model['is_vishing'] == 0].copy()
vishing_df = df_model[df_model['is_vishing'] == 1].copy()

print(legit_df.shape)
print(vishing_df.shape)


(47500, 54)
(2500, 54)


In [10]:

# ====================================
# SDV DATA PREPARATION
# ====================================

def prepare_for_sdv(df_input):

    df_sdv = df_input.copy()

    category_cols = df_sdv.select_dtypes(include=['category']).columns

    for col in category_cols:
        df_sdv[col] = df_sdv[col].astype('object')

    return df_sdv



# Training Strategy

Primero validar pipeline con pocos epochs.
Luego aumentar progresivamente.

Esto evita perder horas por:
- OOM
- errores
- configuraciones inválidas


In [11]:

# ====================================
# SDV METADATA
# ====================================

metadata = SingleTableMetadata()

metadata.detect_from_dataframe(df_model)

metadata


{
    "columns": {
        "session_id": {
            "sdtype": "id"
        },
        "customer_id": {
            "sdtype": "id"
        },
        "session_timestamp": {
            "datetime_format": "%Y-%m-%d %H:%M:%S",
            "sdtype": "datetime"
        },
        "device_type": {
            "sdtype": "categorical"
        },
        "os_type": {
            "sdtype": "categorical"
        },
        "app_version": {
            "sdtype": "categorical"
        },
        "avg_keyhold_ms": {
            "sdtype": "numerical"
        },
        "avg_interkey_latency_ms": {
            "sdtype": "numerical"
        },
        "typing_speed_cps": {
            "sdtype": "numerical"
        },
        "keystroke_variability": {
            "sdtype": "numerical"
        },
        "segmented_typing_ratio": {
            "sdtype": "numerical"
        },
        "avg_touch_pressure": {
            "sdtype": "numerical"
        },
        "avg_touch_size_px": {
            "sdtyp

In [12]:

# ====================================
# FAST CTGAN CONFIG
# ====================================

# --- Smoke-test de arquitectura GPU ---
USE_GPU = False
_gpu_device = None

if torch.cuda.is_available():
    try:
        _t = torch.zeros(4, 4).cuda()
        _ = _t @ _t
        del _t
        torch.cuda.empty_cache()
        USE_GPU = True
        _gpu_device = "cuda"
        name  = torch.cuda.get_device_name(0)
        free, total = torch.cuda.mem_get_info(0)
        print(f"✓ GPU operativa: {name}")
        print(f"  torch {torch.__version__} | VRAM {total/1024**3:.1f} GB ({free/1024**3:.2f} GB libre)")
    except Exception as e:
        print(f"⚠️  GPU smoke-test FALLÓ: {e}")
        print("   Causas posibles:")
        print("   1. No ejecutaste la celda FIX antes del restart")
        print("   2. El acelerador P100 no está activo (Settings > Accelerator)")
        print("   → Continuando en CPU (más lento pero sin error)")
        USE_GPU = False
        _gpu_device = None
else:
    print("⚠️  CUDA no detectado.")
    print("   Activa el acelerador P100 en Settings > Accelerator y reinicia el kernel.")

print(f"\nUSE_GPU = {USE_GPU} | device = {_gpu_device or 'cpu'}")

# batch_size DEBE ser múltiplo de pac (pac=10 por defecto en CTGANSynthesizer)
_batch = 500 if USE_GPU else 500  # 500 es seguro para P100 16GB y también para CPU

ctgan_legit = CTGANSynthesizer(
    metadata=metadata,
    epochs=20,
    batch_size=_batch,
    discriminator_steps=1,
    verbose=True,
    enable_gpu=USE_GPU,
    cuda=_gpu_device,     # explícito: "cuda" o None
)

ctgan_vishing = CTGANSynthesizer(
    metadata=metadata,
    epochs=20,
    batch_size=_batch,
    discriminator_steps=1,
    verbose=True,
    enable_gpu=USE_GPU,
    cuda=_gpu_device,
)

print_resources("POST-INIT CTGAN")


✓ GPU operativa: Tesla T4
  torch 2.10.0+cu128 | VRAM 14.6 GB (14.43 GB libre)

USE_GPU = True | device = cuda
[POST-INIT CTGAN] RAM: 1.5/31.4 GB (6% usado) | VRAM libre: 14.43/14.56 GB


In [13]:
# ====================================
# DIAGNOSTIC: Environment & Synthesizers
# ====================================

import inspect
import sdv

print('sdv.__version__ =', getattr(sdv, '__version__', 'unknown'))
print('torch.__version__ =', torch.__version__)

print('\nConstructor signatures:')
print('CTGANSynthesizer', inspect.signature(CTGANSynthesizer))
print('TVAESynthesizer', inspect.signature(TVAESynthesizer))

print('\nGlobal synth objects available:')
for name in ['ctgan_legit', 'ctgan_vishing', 'tvae']:
    obj = globals().get(name)
    print(f"- {name}: {'present' if obj is not None else 'MISSING'}")

print('\nSynthesizer GPU-related attrs (if present):')
def dump_attrs(obj, obj_name):
    if obj is None:
        return
    for a in ['enable_gpu', 'cuda', 'device']:
        if hasattr(obj, a):
            try:
                val = getattr(obj, a)
                flag = " ✓ GPU" if a == 'enable_gpu' and val is True else (" ✗ CPU" if a == 'enable_gpu' else "")
                print(f"  {obj_name}.{a} = {val}{flag}")
            except Exception as e:
                print(f"  {obj_name}.{a} -> error: {e}")

for name in ['ctgan_legit', 'ctgan_vishing', 'tvae']:
    obj = globals().get(name)
    if obj is not None:
        dump_attrs(obj, name)

print('\nTorch / CUDA info:')
try:
    avail = torch.cuda.is_available()
    print('torch.cuda.is_available() =', avail)
    print('torch.cuda.device_count() =', torch.cuda.device_count())
    if avail:
        print('torch.cuda.get_device_name(0) =', torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info(0)
        print(f'VRAM: {total/1024**3:.1f} GB total, {free/1024**3:.2f} GB libre')
except Exception as e:
    print('CUDA query error:', e)

print('\nEnv vars:')
print('CUDA_VISIBLE_DEVICES   =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('CUDA_LAUNCH_BLOCKING   =', os.environ.get('CUDA_LAUNCH_BLOCKING'))
print('PYTORCH_CUDA_ALLOC_CONF=', os.environ.get('PYTORCH_CUDA_ALLOC_CONF'))

print('\nPython executable:', sys.executable)
print('Platform:', sys.platform)

print('\nRecursos del sistema:')
print_resources("DIAGNOSTIC")

# Smoke-test GPU con tensor pequeño
if USE_GPU:
    try:
        t = torch.randn(100, 100).cuda()
        _ = t @ t.T
        del t
        torch.cuda.empty_cache()
        print('\nSmoke-test GPU: OK (100x100 matmul en CUDA)')
    except Exception as e:
        print(f'\nSmoke-test GPU FALLÓ: {e}')
        print('Revisa que el acelerador P100 esté habilitado en Settings > Accelerator')
else:
    print('\n[ADVERTENCIA] CUDA no disponible. Verifica Settings > Accelerator en Kaggle.')


sdv.__version__ = 1.36.2
torch.__version__ = 2.10.0+cu128

Constructor signatures:
CTGANSynthesizer (metadata, enforce_min_max_values=True, enforce_rounding=True, locales=['en_US'], embedding_dim=128, generator_dim=(256, 256), discriminator_dim=(256, 256), generator_lr=0.0002, generator_decay=1e-06, discriminator_lr=0.0002, discriminator_decay=1e-06, batch_size=500, discriminator_steps=1, log_frequency=True, verbose=False, epochs=300, pac=10, enable_gpu=True, cuda=None)
TVAESynthesizer (metadata, enforce_min_max_values=True, enforce_rounding=True, embedding_dim=128, compress_dims=(128, 128), decompress_dims=(128, 128), l2scale=1e-05, batch_size=500, verbose=False, epochs=300, loss_factor=2, enable_gpu=True, cuda=None)

Global synth objects available:
- ctgan_legit: present
- ctgan_vishing: present
- tvae: MISSING

Synthesizer GPU-related attrs (if present):
  ctgan_legit.enable_gpu = cuda ✗ CPU
  ctgan_vishing.enable_gpu = cuda ✗ CPU

Torch / CUDA info:
torch.cuda.is_available() = True

In [14]:

# ====================================
# TRAIN LEGIT CTGAN
# ====================================

print_resources("PRE-TRAIN LEGIT")

legit_df_sdv = prepare_for_sdv(legit_df)
legit_df_sdv_train = legit_df_sdv.sample(
    n=min(50000, len(legit_df_sdv)),
    random_state=RANDOM_STATE
).copy()

print(f"Entrenando CTGAN legit con {len(legit_df_sdv_train):,} filas...")
print(legit_df_sdv_train.shape)

ctgan_legit.fit(legit_df_sdv_train)

aggressive_gc()
print_resources("POST-TRAIN LEGIT")
print("Legit CTGAN trained ✓")


[PRE-TRAIN LEGIT] RAM: 1.5/31.4 GB (6% usado) | VRAM libre: 14.43/14.56 GB
Entrenando CTGAN legit con 47,500 filas...
(47500, 54)


Gen. (+00.02) | Discrim. (-00.08): 100%|██████████| 20/20 [54:13<00:00, 162.65s/it]


[POST-TRAIN LEGIT] RAM: 2.3/31.4 GB (9% usado) | VRAM libre: 11.29/14.56 GB
Legit CTGAN trained ✓


In [15]:

# ====================================
# TRAIN VISHING CTGAN
# ====================================

print_resources("PRE-TRAIN VISHING")

vishing_df_sdv = prepare_for_sdv(vishing_df)
print(f"Entrenando CTGAN vishing con {len(vishing_df_sdv):,} filas...")

ctgan_vishing.fit(vishing_df_sdv)

aggressive_gc()
print_resources("POST-TRAIN VISHING")
print("Vishing CTGAN trained ✓")


[PRE-TRAIN VISHING] RAM: 2.4/31.4 GB (9% usado) | VRAM libre: 11.29/14.56 GB
Entrenando CTGAN vishing con 2,500 filas...


Gen. (+01.01) | Discrim. (+00.10): 100%|██████████| 20/20 [00:11<00:00,  1.78it/s]


[POST-TRAIN VISHING] RAM: 3.8/31.4 GB (14% usado) | VRAM libre: 11.19/14.56 GB
Vishing CTGAN trained ✓


In [16]:

# ====================================
# TVAE  (GPU enabled)
# ====================================

print_resources("PRE-TRAIN TVAE")

df_model_sdv = prepare_for_sdv(df_model)

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=15,
    batch_size=512,
    verbose=True,
    enable_gpu=USE_GPU,
    cuda=_gpu_device,
)

print(f"Entrenando TVAE con {len(df_model_sdv):,} filas...")
tvae.fit(df_model_sdv)

aggressive_gc()
print_resources("POST-TRAIN TVAE")
print("TVAE trained ✓")


[PRE-TRAIN TVAE] RAM: 3.8/31.4 GB (14% usado) | VRAM libre: 11.19/14.56 GB
Entrenando TVAE con 50,000 filas...


Loss: +19.13: 100%|██████████| 15/15 [02:03<00:00,  8.20s/it]


[POST-TRAIN TVAE] RAM: 4.0/31.4 GB (14% usado) | VRAM libre: 11.19/14.56 GB
TVAE trained ✓


In [24]:

# ====================================
# CONSTRAINT ENGINE
# ====================================

def apply_constraints(df_aug):

    if 'phone_call_active' in df_aug.columns and 'call_overlap_duration_s' in df_aug.columns:

        mask = df_aug['phone_call_active'] == 0

        df_aug.loc[mask, 'call_overlap_duration_s'] = 0

    if 'transaction_attempted' in df_aug.columns and 'transaction_amount_cop' in df_aug.columns:

        mask = df_aug['transaction_attempted'] == 0

        df_aug.loc[mask, 'transaction_amount_cop'] = 0

    if 'dead_time_ratio' in df_aug.columns:

        df_aug['dead_time_ratio'] = np.clip(
            df_aug['dead_time_ratio'],
            0,
            1
        )

    if 'segmented_typing_ratio' in df_aug.columns:

        df_aug['segmented_typing_ratio'] = np.clip(
            df_aug['segmented_typing_ratio'],
            0,
            1
        )

    return df_aug


In [25]:

# ====================================
# HARD CASE GENERATION
# ====================================

def generate_hard_legit(df_input, n):

    hard = df_input.sample(
        n=n,
        replace=True
    ).copy()

    if 'phone_call_active' in hard.columns:
        hard['phone_call_active'] = np.random.binomial(
            1,
            0.35,
            size=n
        )

    if 'hesitation_count' in hard.columns:
        hard['hesitation_count'] *= np.random.uniform(
            1.2,
            1.8,
            size=n
        )

    hard['is_vishing'] = 0

    return hard

def generate_soft_vishing(df_input, n):

    soft = df_input.sample(
        n=n,
        replace=True
    ).copy()

    if 'hesitation_count' in soft.columns:
        soft['hesitation_count'] *= np.random.uniform(
            0.5,
            0.8,
            size=n
        )

    soft['is_vishing'] = 1

    return soft


In [26]:

# ====================================
# PARQUET STREAMING WRITER
# ====================================

parquet_writer = None

def append_parquet(df_chunk):

    global parquet_writer

    table = pa.Table.from_pandas(df_chunk)

    if parquet_writer is None:

        parquet_writer = pq.ParquetWriter(
            OUTPUT_PARQUET,
            table.schema,
            compression='snappy'
        )

    parquet_writer.write_table(table)



# Chunked Generation

La generación chunked evita:
- RAM overflow
- copies gigantes
- crashes de Kaggle

Cada chunk:
- se genera
- se valida
- se exporta
- se libera de memoria


In [27]:

# ====================================
# GENERATE LEGIT CHUNKS
# ====================================

print_resources("INICIO GENERACIÓN LEGIT")
remaining_legit = TARGET_LEGIT
chunk_num = 0

while remaining_legit > 0:

    # Guardia de memoria RAM
    if ram_free_gb() < RAM_LOW_THRESHOLD_GB:
        print(f"[AVISO] RAM libre crítica ({ram_free_gb():.2f} GB). Esperando GC...")
        aggressive_gc()
        import time; time.sleep(3)
        if ram_free_gb() < RAM_LOW_THRESHOLD_GB:
            print("[ERROR] RAM insuficiente. Deteniendo generación para evitar OOM.")
            break

    # Advertencia de VRAM
    if USE_GPU and vram_free_gb() < VRAM_LOW_THRESHOLD_GB:
        print(f"[AVISO] VRAM libre baja ({vram_free_gb():.2f} GB). Vaciando caché CUDA...")
        aggressive_gc()

    current_chunk = min(CHUNK_SIZE, remaining_legit)
    chunk_num += 1

    ctgan_chunk = ctgan_legit.sample(num_rows=int(current_chunk * 0.7))
    tvae_chunk = tvae.sample(num_rows=int(current_chunk * 0.2))
    bootstrap_chunk = legit_df.sample(n=int(current_chunk * 0.1), replace=True).copy()
    hard_legit = generate_hard_legit(legit_df, n=max(1000, int(current_chunk * 0.02)))

    chunk = pd.concat([ctgan_chunk, tvae_chunk, bootstrap_chunk, hard_legit], ignore_index=True)
    chunk['is_vishing'] = 0
    chunk = apply_constraints(chunk)
    chunk = optimize_memory(chunk)

    append_parquet(chunk)

    remaining_legit -= len(chunk)
    print(f"[Chunk {chunk_num}] Remaining legit: {remaining_legit:,}", end="  ")
    print_resources()

    del ctgan_chunk, tvae_chunk, bootstrap_chunk, hard_legit, chunk
    aggressive_gc()

print("\nGeneración legit completa.")


[INICIO GENERACIÓN LEGIT] RAM: 4.0/31.4 GB (14% usado) | VRAM libre: 11.19/14.56 GB
[Chunk 1] Remaining legit: 888,000  RAM: 4.0/31.4 GB (14% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 2] Remaining legit: 786,000  RAM: 2.6/31.4 GB (10% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 3] Remaining legit: 684,000  RAM: 2.6/31.4 GB (10% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 4] Remaining legit: 582,000  RAM: 2.6/31.4 GB (10% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 5] Remaining legit: 480,000  RAM: 2.7/31.4 GB (10% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 6] Remaining legit: 378,000  RAM: 2.6/31.4 GB (10% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 7] Remaining legit: 276,000  RAM: 2.6/31.4 GB (10% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 8] Remaining legit: 174,000  RAM: 2.6/31.4 GB (10% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 9] Remaining legit: 72,000  RAM: 2.7/31.4 GB (10% usado) | VRAM libre: 10.70/14.56 GB
[Chunk 10] Remaining legit: -1,440  RAM: 2.7/31.4 GB (10% usado) | VRAM

In [28]:

# ====================================
# GENERATE VISHING CHUNKS
# ====================================

print_resources("INICIO GENERACIÓN VISHING")
remaining_vishing = TARGET_VISHING
chunk_num = 0

while remaining_vishing > 0:

    if ram_free_gb() < RAM_LOW_THRESHOLD_GB:
        print(f"[AVISO] RAM libre crítica ({ram_free_gb():.2f} GB). Esperando GC...")
        aggressive_gc()
        import time; time.sleep(3)
        if ram_free_gb() < RAM_LOW_THRESHOLD_GB:
            print("[ERROR] RAM insuficiente. Deteniendo generación para evitar OOM.")
            break

    if USE_GPU and vram_free_gb() < VRAM_LOW_THRESHOLD_GB:
        print(f"[AVISO] VRAM libre baja ({vram_free_gb():.2f} GB). Vaciando caché CUDA...")
        aggressive_gc()

    current_chunk = min(20_000, remaining_vishing)
    chunk_num += 1

    ctgan_chunk = ctgan_vishing.sample(num_rows=int(current_chunk * 0.6))
    tvae_chunk = tvae.sample(num_rows=int(current_chunk * 0.2))
    bootstrap_chunk = vishing_df.sample(n=int(current_chunk * 0.2), replace=True).copy()
    soft_vishing = generate_soft_vishing(vishing_df, n=max(500, int(current_chunk * 0.05)))

    chunk = pd.concat([ctgan_chunk, tvae_chunk, bootstrap_chunk, soft_vishing], ignore_index=True)
    chunk['is_vishing'] = 1
    chunk = apply_constraints(chunk)
    chunk = optimize_memory(chunk)

    append_parquet(chunk)

    remaining_vishing -= len(chunk)
    print(f"[Chunk {chunk_num}] Remaining vishing: {remaining_vishing:,}", end="  ")
    print_resources()

    del ctgan_chunk, tvae_chunk, bootstrap_chunk, soft_vishing, chunk
    aggressive_gc()

print("\nGeneración vishing completa.")


[INICIO GENERACIÓN VISHING] RAM: 2.7/31.4 GB (10% usado) | VRAM libre: 11.19/14.56 GB
[Chunk 1] Remaining vishing: -500  RAM: 2.7/31.4 GB (10% usado) | VRAM libre: 11.07/14.56 GB

Generación vishing completa.


In [29]:

# ====================================
# CLOSE WRITER
# ====================================

if parquet_writer:
    parquet_writer.close()

print("Parquet export completed")


Parquet export completed


In [30]:

# ====================================
# LOAD FINAL DATASET
# ====================================

final_df = pd.read_parquet(OUTPUT_PARQUET)

print(final_df.shape)

final_df.head()


(1001940, 54)


,session_id,customer_id,session_timestamp,device_type,os_type,app_version,avg_keyhold_ms,avg_interkey_latency_ms,typing_speed_cps,keystroke_variability,...,remote_access_tool_detected,suspicious_app_detected,transaction_attempted,transaction_amount_cop,is_new_beneficiary,time_to_transaction_s,errors_per_minute,interactions_per_s,hesitation_composite,is_vishing
0,sdv-id-QZWShb,CUS-72848,2024-10-01 01:24:06,mobile,Android,v12.1,77.963791,165.405869,4.807504,0.333306,...,0,0,0,0,0,10.0,8.717,2.4,1.151666,0
1,sdv-id-PEoYCd,CUS-61785,2025-05-15 10:50:29,mobile,Android,v12.2,75.956017,187.874390,5.676887,0.234562,...,0,0,1,0,0,10.0,8.717,2.4,1.150960,0
2,sdv-id-ujvcPk,CUS-49773,2024-12-16 23:45:55,mobile,Android,v12.1,108.780571,109.286858,2.531858,0.097838,...,0,0,1,865,1,0.0,8.717,2.4,1.151046,0
3,sdv-id-KIPQgn,CUS-51647,2024-08-11 03:36:09,mobile,iOS,v13.0,81.211784,193.334930,2.252960,0.310742,...,0,0,1,703930,0,10.0,8.717,2.4,1.150729,0
4,sdv-id-IhVGKN,CUS-75231,2025-01-28 03:02:56,mobile,Android,v12.1,99.984436,131.283447,4.983382,0.222195,...,0,0,1,1660352,0,0.0,8.717,2.4,1.151636,0


In [31]:

# ====================================
# ADVERSARIAL VALIDATION
# ====================================

orig = df_model.sample(
    n=min(50000, len(df_model)),
    random_state=42
).copy()

orig['synthetic'] = 0

synth = final_df.sample(
    n=min(150000, len(final_df)),
    random_state=42
).copy()

synth['synthetic'] = 1

adv_df = pd.concat([orig, synth])

drop_cols = [
    c for c in [
        'session_id',
        'customer_id',
        'session_timestamp'
    ]
    if c in adv_df.columns
]

X = adv_df.drop(columns=['synthetic'] + drop_cols)

for col in X.select_dtypes(include=['object', 'category']).columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

X = X.fillna(0)

y = adv_df['synthetic']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

clf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

clf.fit(X_train, y_train)

preds = clf.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, preds)

print(f"Adversarial Validation AUC: {auc:.4f}")


Adversarial Validation AUC: 0.9398


In [32]:

# ====================================
# FINAL STATS
# ====================================

print(final_df['is_vishing'].value_counts())

print(final_df['is_vishing'].value_counts(normalize=True))

print(final_df.memory_usage(deep=True).sum() / 1024**2, "MB")


is_vishing
0    991440
1     10500
Name: count, dtype: int64
is_vishing
0    0.98952
1    0.01048
Name: proportion, dtype: float64
314.0548257827759 MB



# Recomendaciones de Ejecución

## Primera corrida

NO generar 1M inicialmente.

Usar:

```python
TARGET_ROWS = 100_000
```

para validar.

---

## Luego

Escalar progresivamente:

- 250k
- 500k
- 1M

---

## Configuración recomendada Kaggle

- GPU P100/T4
- Internet OFF
- RAM High

---

## Tiempo esperado

| Etapa | Tiempo |
|---|---|
| Training | 30-60 min |
| Generation | 10-30 min |
| Validation | 5-15 min |

Tiempo total esperado:

~45 min a 1.5h
